<a href="https://colab.research.google.com/github/ssuwkang/ssubiochem/blob/main/Boltz-2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 🧬 Boltz-2 원클릭 단백질 구조 및 결합 친화도 예측기
#@markdown ---
#@markdown ### 1. 작업 및 분자 정보 입력
#@markdown 원하는 값을 입력한 뒤, **좌측 상단 재생 버튼(`▶`)**을 한 번만 누르세요. (설치부터 결과 다운로드까지 자동 진행됩니다)

jobname = "EGFR.Lazertinib" #@param {type:"string"}
#@markdown **단백질 아미노산 서열** (복합체는 `:` 기호로 사슬 구분)
protein_sequences = "MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFEDHFLSLQRMFNNCEVVLGNLEITYVQRNYDLSFLKTIQEVAGYVLIALNTVERIPLENLQIIRGNMYYENSYALAVLSNYDANKTGLKELPMRNLQEILHGAVRFSNNPALCNVESIQWRDIVSSDFLSNMSMDFQNHLGSCQKCDPSCPNGSCWGAGEENCQKLTKIICAQQCSGRCRGKSPSDCCHNQCAAGCTGPRESDCLVCRKFRDEATCKDTCPPLMLYNPTTYQMDVNPEGKYSFGATCVKKCPRNYVVTDHGSCVRACGADSYEMEEDGVRKCKKCEGPCRKVCNGIGIGEFKDSLSINATNIKHFKNCTSISGDLHILPVAFRGDSFTHTPPLDPQELDILKTVKEITGFLLIQAWPENRTDLHAFENLEIIRGRTKQHGQFSLAVVSLNITSLGLRSLKEISDGDVIISGNKNLCYANTINWKKLFGTSGQKTKIISNRGENSCKATGQVCHALCSPEGCWGPEPRDCVSCRNVSRGRECVDKCNLLEGEPREFVENSECIQCHPECLPQAMNITCTGRGPDNCIQCAHYIDGPHCVKTCPAGVMGENNTLVWKYADAGHVCHLCHPNCTYGCTGPGLEGCPTNGPKIPSIATGMVGALLLLLVVALGIGLFMRRRHIVRKRTLRRLLQERELVEPLTPSGEAPNQALLRILKETEFKKIKVLGSGAFGTVYKGLWIPEGEKVKIPVAIKELREATSPKANKEILDEAYVMASVDNPHVCRLLGICLTSTVQLITQLMPFGCLLDYVREHKDNIGSQYLLNWCVQIAKGMNYLEDRRLVHRDLAARNVLVKTPQHVKITDFGLAKLLGAEEKEYHAEGGKVPIKWMALESILHRIYTHQSDVWSYGVTVWELMTFGSKPYDGIPASEISSILEKGERLPQPPICTIDVYMIMVKCWMIDADSRPKFRELIIEFSKMARDPQRYLVIQGDERMHLPSPTDSNFYRALMDEEDMDDVVDADEYLIPQQGFFSSPSTSRTPLLSSLSATSNNSTVACIDRNGLQSCPIKEDSFLQRYSSDPTGALTEDSIDDTFLPVPEYINQSVPKRPAGSVQNPVYHNQPLNPAPSRDPHYQDPHSTAVGNPEYLNTVQPTCVNSTFDSPAHWAQKGSHQISLDNPDYQQDFFPKEAKPNGIFKGSTAENAEYLRVAPQSSEFIGA" #@param {type:"string"}
#@markdown **저분자 리간드 SMILES** (선택 사항 - 비워두면 단백질 구조만 예측)
ligand_smiles = "CN(C)CC1=CN(N=C1C2=CC=CC=C2)C3=NC(=NC=C3)NC4=C(C=C(C(=C4)NC(=O)C=C)N5CCOCC5)OC" #@param {type:"string"}
#@markdown **결합 친화도(Affinity) 예측 포함 여부** (리간드 입력 시 적용)
predict_affinity = True #@param {type:"boolean"}
#@markdown **예측 완료 후 결과 ZIP 파일 자동 다운로드**
auto_download = True #@param {type:"boolean"}

# ==============================================================================
# 이하 실행 파이프라인 (수정 불필요)
# ==============================================================================
import os
import sys
import glob
import json
import shutil
import importlib.util
import subprocess

print("==================================================")
print("🚀 [1/4] 실행 환경 및 Boltz-2 설치 상태 확인 중...")
print("==================================================")

# GPU 연결 여부 확인
gpu_check = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu_check.returncode != 0:
    raise RuntimeError("❌ GPU가 활성화되지 않았습니다! 상단 메뉴 [런타임] -> [런타임 유형 변경]에서 'T4 GPU'를 선택 후 다시 실행하세요.")

# 이미 설치되어 있으면 건너뛰고, 미설치 시에만 자동 설치
if importlib.util.find_spec("boltz") is None or importlib.util.find_spec("py3Dmol") is None:
    print("📦 필수 패키지를 설치합니다 (최초 1회, 약 1분 소요)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "boltz[cuda]", "-U", "py3Dmol", "pyyaml"], check=True)
    print("✅ 패키지 설치 완료!")
else:
    print("✅ 이미 패키지가 설치되어 있어 설치 단계를 건너뜁니다.")

import yaml
import py3Dmol
from google.colab import files

print("\n==================================================")
print("📝 [2/4] 입력 데이터 검증 및 YAML 설정 생성 중...")
print("==================================================")

jobname_clean = jobname.strip() or "boltz2_run"
os.makedirs(jobname_clean, exist_ok=True)
yaml_data = {"version": 1, "sequences": []}

chains = [seq.strip().upper() for seq in protein_sequences.split(":") if seq.strip()]
if not chains:
    raise ValueError("❌ 단백질 서열이 비어 있습니다. 아미노산 서열을 입력해 주세요.")

smiles_clean = ligand_smiles.strip()
total_entities = len(chains) + (1 if smiles_clean else 0)
if total_entities > 26:
    raise ValueError("❌ 단백질 사슬과 리간드를 합쳐 최대 26개(A~Z)까지만 지원합니다.")

chain_ids = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
for idx, seq in enumerate(chains):
    yaml_data["sequences"].append({
        "protein": {
            "id": chain_ids[idx],
            "sequence": seq
        }
    })

if smiles_clean:
    ligand_id = chain_ids[len(chains)]
    yaml_data["sequences"].append({
        "ligand": {
            "id": ligand_id,
            "smiles": smiles_clean
        }
    })
    if predict_affinity:
        yaml_data["properties"] = [
            {"affinity": {"binder": ligand_id}}
        ]

yaml_path = os.path.join(jobname_clean, f"{jobname_clean}.yaml")
with open(yaml_path, "w") as f:
    yaml.dump(yaml_data, f, sort_keys=False)

mode_str = "단백질 단독" if len(chains) == 1 else f"단백질 복합체 ({len(chains)}개 사슬)"
if smiles_clean:
    mode_str += f" + 저분자 리간드(Chain {ligand_id})" + (" [결합 친화도 포함]" if predict_affinity else "")
print(f"📌 실행 모드: {mode_str}")

print("\n==================================================")
print("🧠 [3/4] Boltz-2 AI 구조 예측 실행 중...")
print("⏳ 첫 실행 시 모델 가중치(약 6GB) 다운로드로 2~3분 추가 소요됩니다.")
print("==================================================")

out_dir = os.path.join(jobname_clean, "output")
# 실시간 로그 출력을 위해 Popen 스트리밍 사용
process = subprocess.Popen(
    ["boltz", "predict", yaml_path, "--use_msa_server", "--use_potentials", "--out_dir", out_dir],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)
for line in process.stdout:
    print(line, end="")
process.wait()

if process.returncode != 0:
    raise RuntimeError("❌ Boltz-2 예측 실행 중 오류가 발생했습니다. 위 로그를 확인하세요.")

print("\n==================================================")
print("📊 [4/4] 예측 결과 분석 및 3D 구조 시각화")
print("==================================================")

cif_files = sorted(glob.glob(f"{out_dir}/**/*model_0.cif", recursive=True))
if not cif_files:
    cif_files = sorted(glob.glob(f"{out_dir}/**/*.cif", recursive=True))
if not cif_files:
    raise FileNotFoundError("❌ 생성된 .cif 구조 파일을 찾을 수 없습니다.")

top_cif = cif_files[0]
print(f"🧬 1순위 예측 구조 파일: {top_cif}")

conf_files = sorted(glob.glob(f"{out_dir}/**/confidence_*model_0.json", recursive=True))
if conf_files:
    with open(conf_files[0]) as f:
        conf = json.load(f)
    print(f"📈 구조 신뢰도 지표: pTM = {conf.get('ptm', 0):.3f} | ipTM = {conf.get('iptm', 0):.3f} | Complex pLDDT = {conf.get('complex_plddt', 0):.3f}")

aff_files = sorted(glob.glob(f"{out_dir}/**/affinity_*.json", recursive=True))
if aff_files:
    with open(aff_files[0]) as f:
        aff = json.load(f)
    prob = float(aff.get("affinity_probability_binary", 0.0))
    log_ic50 = float(aff.get("affinity_pred_value", 0.0))
    pic50 = 6.0 - log_ic50
    delta_g = (6.0 - pic50) * 1.364
    print("💊 저분자 결합 예측 결과 (모델 예측치 기반 근사):")
    print(f"   • 결합 확률 (Binding Probability): {prob * 100:.1f}%")
    print(f"   • 예측 친화도: log10(IC50) = {log_ic50:.3f} (μM) | pIC50 ≈ {pic50:.2f} | ΔG ≈ {delta_g:.2f} kcal/mol")

# 3D 뷰어 렌더링
view = py3Dmol.view(width=750, height=500)
with open(top_cif, "r") as f:
    view.addModel(f.read(), "cif")

view.setStyle({"polymer": True}, {"cartoon": {"color": "spectrum"}})
view.setStyle({"hetflag": True}, {"stick": {"colorscheme": "greenCarbon", "radius": 0.3}})
view.zoomTo()
view.update()
view.show()

if auto_download:
    zip_name = f"{jobname_clean}_boltz2_results"
    archive_path = shutil.make_archive(zip_name, "zip", jobname_clean)
    print(f"\n📥 전체 결과 압축 파일 생성 완료: {archive_path}")
    try:
        files.download(archive_path)
    except Exception as e:
        print(f"⚠️ 자동 다운로드가 차단되었을 수 있습니다. 좌측 파일 탐색기에서 '{zip_name}.zip'을 직접 다운로드하세요. ({e})")

print("\n🎉 모든 작업이 성공적으로 완료되었습니다!")

🚀 [1/4] 실행 환경 및 Boltz-2 설치 상태 확인 중...
📦 필수 패키지를 설치합니다 (최초 1회, 약 1분 소요)...
